<a href="https://colab.research.google.com/github/angelgreen/paperAndNote/blob/master/Transformer%E6%95%B0%E5%AD%A65.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# FFN
接上文，我们都是在讲token如何“感知”其他token。本章我们讲，如何学习“知识”。
FFN 负责对每个位置上的表示进行非线性变换，是模型进行特征抽象和知识存储的重要组件。

## Position-wise Feed-Forward Networks
在Attention is all you need的论文中，作者提出了一个简单的全连接层，对每个token分别独立并且相同的处理。

例如，输入$X$序列：
$$
  X \in \mathbb{R}^{B\times T \times d_{model}}，\quad x_t \in \mathbb{R}^{d_{model}}
$$
其中，
$x_t$表示某个样本在第$t$个位置上的 Token 表示。

实际上：

第一个token:
$$
  x_1
$$

第二个token:
$$
  x_2
$$
都会经过：
$$
  FFN(x_i)
$$
每个token在FFN计算时，都是共享参数的。

通俗一点，Attention负责“token之间聊天”，FFN负责“每个token自己思考”。这里的“自己思考”并不是说Token不包含上下文信息，经过Attention后，每个Token的表示已经融合了其他Token的信息，FFN则在此基础上，对每个位置的表示进行独立的非线性变换。

原始的Transformer论文中的FFN定义为：
$$
  \operatorname{FFN}(X) = \max(0, XW_1 + b_1)W_2 + b_2
$$
其中，$X \in \mathbb{R}^{B\times T \times d_{model}}$，$W_1 \in \mathbb{R}^{d_{model} \times d_{ff}}$，$W_2 \in \mathbb{R}^{d_{ff} \times d_{model}}$。一般来说，$d_{ff} \gg d_{model}$。其实，这也是一种“哲学”，先升维空间，在该空间进行激活，然后降维回到原始空间。

比如：
$$
  d_{model} = 512
$$
升维：
$$
  512 \rightarrow 2048
$$
在高维空间：
$$
  \operatorname{ReLU}(xW_1)
$$
学习复杂模式。

最后：
$$
  2048 \rightarrow 512
$$
压缩回原空间。

中间层扩展提供了更宽的特征空间，配合非线性激活，网络就能学习更丰富的特征组合。
中间的非线性激活很重要。如果只有两次线性变换，中间没有非线性，那么两层线性变换可以合并为一次线性变换，无法获得同样的非线性表达能力。

## 从ReLU到GELU
我们知道ReLU激活函数的定义为：
$$
 \operatorname{ReLU}(x) = \left\{
  \begin{aligned}
    x \quad, x >  0 \\
    0 \quad, x \le 0 \\
  \end{aligned}
  \right.
$$
有什么问题呢？如果神经网络参数更新了，当某个神经元的输入持续处于负半轴时，ReLU的梯度为零，该神经元对应的参数可能无法通过这些输入得到有效的更新；如果这种状态长期持续，就可能出现所谓的“死亡ReLU”问题。为了探索更加平滑的非线性变换，研究者提出了GELU、Swish等激活函数，它们不会像ReLU一样直接将整个负半轴截断为零，从而为网络提供了不同的梯度和特征变换特性。引入新的GELU/swish激活函数，
$$
  \operatorname{GELU}(x) = x \cdot \Phi(x) = x \cdot P(Z \le x), \text{Where}\;  Z \sim \mathcal{N}(0,1)
$$

GELU激活函数相比ReLU激活函数，有如下几个好处：
1. 平滑性：GELU 在负半轴不会像 ReLU 那样直接输出零，因此负输入也可能传递非零梯度。不过，这不意味着 GELU 能彻底解决所有梯度问题。
2. 概率加权的直观解释：$\Phi(x)$对输入进行连续加权。对于较大的正输入，$\Phi(x)$接近1，因此GELU的输出接近原始输入；对于负输入，输出通常受到抑制，但不会像ReLU那样将整个负半轴直接截断为零。

## 从普通FFN到GLU/SwiGLU


后面，又引入Gated Linear Units and Variants，引入“门控”激活，通过自学习，来控制需要学习哪些“知识”：
$$
\begin{aligned}
  \operatorname{FFN_{GEGLU}}(x) & = (\operatorname{GELU}(xW_g) \odot xW_u) W_d \\
  \operatorname{FFN_{SwiGLU}}(x) & = (\operatorname{SiLU}(xW_g) \odot xW_u) W_d \\
\end{aligned}
$$
其中，$\odot$表示逐元素乘法；$W_g$与$W_u$分别对应门控分支和另一条信息分支，$W_d$将中间表示映射回模型维度。

这里采用的是常见的SwiGLU形式，其中$\operatorname{SiLU}(x) = x\sigma(x)$。

传统FFN通过激活函数进行非线性变换；GLU类结构则引入一个可学习的门控分支，让网络根据当前输入动态调节另一条分支的信息。


## 提升模型的容量，却不让每个token的计算量同比增长
通常情况下，增加模型参数容量，可以为模型提供学习和表示更多复杂模式的潜力，但不意味着模型能力一定随着参数量同步增加。如果希望增加Dense FFN的参数容量，一个直接的方法是扩大中间层维度$d_ff$。但由于所有 Token 都需要经过同一个 FFN，扩大中间层也会增加每个 Token 的计算量。那么，有没有另一种方式，让模型拥有更多参数，却不必让每个 Token 都执行全部新增的计算？

### Mixture of Experts （混合专家网络）
混合专家网络（Mixture of Experts, MoE）将原来的单个 FFN 替换为多个并行的 Expert 子网络，并通过 Router 为不同 Token 动态选择需要激活的 Expert。主要包括四大组件：
1. lightweight Router，动态决定每个 Token 应该由哪些 Expert 处理。
2. Token Dispatcher， 根据 Router 的选择，将 Token 按目标 Expert 进行分组和重排，并在 Expert 分布于不同设备时，负责相应的跨设备分发。
3. Expert FFN,    每个Experts中的FFN。
4. Combine，      收集Experts处理后的token，并还原到原始序列的位置。

让我们一步步展开MoE层，（为了方便介绍，我们假设使用Top-1选择）

第一步，Router： 选择Expert
$$
  r(x) = xW_r, \quad p(x) = \operatorname{Softmax}(r(x))
$$
其中：
$$
  W_r \in \mathbb{R}^{d_{model}\times N}
$$
$N$为Expert的数量。
根据路由概率选择Top-1个Expert。

第二步，Dispatcher：分发Token
将Token按目标Expert重新排列，再发送给对应的计算单元。只有Expert分布在不同设备上时，才需要相应的跨设备通信。

第三步，Expert Computation：独立计算
每个Expert使用自己的FFN参数处理接收到的Token，
$$
  \operatorname{FFN_{SwiGLU}}(x) = (\operatorname{SiLU}(xW_g) \odot xW_u) W_d
$$

第四步，Combine：合并输出
将Expert输出乘以对应的路由权重，再按照原始Token的位置恢复顺序。

最终，
$$
  \operatorname{MoE}(x) = \sum_{i\in S(x)} g_i(x) E_i(x)
$$
其中，$S(x)$是选中的Expert集合，$g_i(x)$是对应的路由权重。

### 计算量：为什么增加Expert，不必让每个Token都付出更多的计算？
假设原来的Dense FFN有1份参数，现在我们准备了8个Expert，并且每个Expert的规模与原来的Dense FFN相同。

在Top-1路由下，每个Token只会被分配给其中一个Expert。
| 对比项        | Dense FFN | 8-Expert MoE |
|------------------------|--------------|-----------------|
|Expert部分参数量      | 1份    | 8份      |
|每个Token激活的Expert | 1个共享FFN | 1 个 Expert |
|每个Token的主要FFN计算量| 约1份   | 约 1份    |

可以看到，MoE 的 Expert 总参数量增加了，但每个 Token 并不需要执行全部 8 个 Expert 的计算。

因此，MoE 为模型提供了更大的参数容量，同时让每个 Token 的主要 FFN 计算量仍然与原来的 Dense FFN 大致处于同一水平。

当然，这里只比较了 FFN/Expert 部分的主要计算。Router 本身也需要计算，实际系统还要处理 Token 分发、结果合并和可能的跨设备通信，因此总推理开销并不会完全保持不变。

# 最后
回顾一下，从普通 FFN 到 GELU，再到 SwiGLU，我们看到的是 FFN 内部计算方式的演化：模型不仅能够通过非线性激活变换特征，还能借助门控分支，根据当前输入动态调节不同信息的作用。

而 MoE 则从另一个角度改变了 FFN 的组织方式：与其让所有 Token 都经过同一个越来越大的 FFN，不如准备多个 Expert，再根据每个 Token 的隐藏状态，动态选择需要激活的部分网络。

**Dense 的思路是让所有 Token 共用更大的 FFN，而 MoE 的思路是让不同 Token 有条件地使用不同 Expert。**

当然，这种稀疏计算并非没有代价。Router 如何合理分配 Token，不同 Expert 如何保持负载均衡，以及跨 GPU 时如何高效地进行 Token 分发和结果合并，都是 MoE 在实际应用中需要解决的问题。

到这里，我们已经分别了解了 Attention 和 FFN：前者负责 Token 之间的信息交互，后者负责对每个 Token 的表示进行进一步加工。

那么，当这两个组件组合在一起时，一个完整的 Transformer Block 究竟是如何工作的？残差连接、LayerNorm 与 Attention、FFN 又是如何协同工作的？

下一篇，我们将回到全局视角，重新审视 Transformer 的整体结构，把前面学到的知识真正串起来。

<!-- 开始今天的活 -->